# Digital Consumer Interest in Indonesian E-Commerce
### Google Trends Analysis of Temporal and Regional Search Patterns, 2020–2025

Notebook ini adalah versi audit untuk dijalankan di Google Colab. Jalankan sel secara berurutan dari atas ke bawah.

**Ruang lingkup:** analisis pola pencarian Google Trends untuk Shopee, Tokopedia, Lazada, Blibli, dan TikTok Shop. Indeks pencarian bukan data penjualan atau pangsa pasar.

**Catatan interpretasi:** nilai Google Trends 0–100 bersifat relatif. Jika tiap platform diunduh pada kueri terpisah, perbandingan level antarplatform tidak sahih. Karena itu, hasil lintas-platform dan label “platform dominan” harus ditafsirkan hati-hati.

**Penting:** periksa kembali tanggal peristiwa dan sumber Google Trends sebelum menjadikan hasil sebagai temuan final.

## 0. Persiapan

In [ ]:
# Instalasi library (jalankan sekali di Google Colab)
%pip install -q pandas numpy matplotlib seaborn scipy statsmodels pymannkendall ruptures openpyxl

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pymannkendall as mk
import ruptures as rpt
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.multitest import multipletests
from itertools import combinations

warnings.filterwarnings("default")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

# ---------------- KONFIGURASI ----------------
DATA_PATH = "/content/Dataset E-Commerce.xlsx"
OUT_DIR = "/content/output_analisis"
PLATFORMS = ["Shopee", "Tokopedia", "Lazada", "Blibli", "TikTok Shop"]
COLORS = dict(zip(PLATFORMS, ["#EE4D2D", "#42B549", "#0F146D", "#0095DA", "#111111"]))
TIKTOK_START = pd.Timestamp("2021-09-01")
EVENTS = {
    "Larangan social commerce (Permendag 31/2023)": "2023-10-01",
    "Integrasi Tokopedia–TikTok Shop": "2023-12-01",
}
os.makedirs(OUT_DIR, exist_ok=True)

def savefig(name):
    plt.savefig(os.path.join(OUT_DIR, name), bbox_inches="tight", dpi=200)

print("Konfigurasi siap.")
print("Dataset:", DATA_PATH)
print("Folder output:", OUT_DIR)

## 1. Memuat dan Membersihkan Data

In [ ]:
# ==========================================
# 1. MEMUAT DAN MEMVALIDASI DATASET
# ==========================================

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"File tidak ditemukan: {DATA_PATH}\n"
        "Di Google Colab, unggah 'Dataset E-Commerce.xlsx' melalui panel Files ke /content."
    )

def load_sheet(sheet_name):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet_name, header=2)
    df = df.dropna(how="all").dropna(axis=1, how="all")
    df.columns = df.columns.astype(str).str.strip()
    df = df.rename(columns={"Tiktok Shop": "TikTok Shop", "Tiktok shop": "TikTok Shop"})
    return df

def validate_platform_columns(df, sheet_name):
    missing = [c for c in PLATFORMS if c not in df.columns]
    if missing:
        raise ValueError(f"Kolom platform hilang di sheet {sheet_name}: {missing}")

# A. Time Series
ts_raw = load_sheet("Time Series")
validate_platform_columns(ts_raw, "Time Series")
if "Kode Periode" not in ts_raw.columns:
    raise ValueError("Kolom 'Kode Periode' tidak ditemukan di Time Series.")

ts_raw["Kode Periode"] = pd.to_datetime(ts_raw["Kode Periode"], errors="coerce")
if ts_raw["Kode Periode"].isna().any():
    raise ValueError("Ada tanggal kosong/tidak valid di Time Series.")
if ts_raw["Kode Periode"].duplicated().any():
    raise ValueError("Ada tanggal duplikat di Time Series.")
for col in PLATFORMS:
    ts_raw[col] = pd.to_numeric(ts_raw[col], errors="coerce")
if ts_raw[PLATFORMS].isna().any().any():
    raise ValueError("Ada nilai kosong atau nonnumerik di kolom platform Time Series.")

ts = ts_raw.set_index("Kode Periode").sort_index()
expected_dates = pd.date_range("2020-01-01", "2025-12-01", freq="MS")
if not ts.index.equals(expected_dates):
    raise ValueError("Tanggal Time Series tidak persis mencakup 72 bulan Jan 2020–Des 2025. Periksa data.")
if not ts[PLATFORMS].apply(lambda col: col.between(0, 100).all()).all():
    raise ValueError("Ada nilai indeks Time Series di luar rentang 0–100.")
ts.index.freq = "MS"

# B. Regional
reg_raw = load_sheet("Regional")
validate_platform_columns(reg_raw, "Regional")
if "Wilayah" not in reg_raw.columns:
    raise ValueError("Kolom 'Wilayah' tidak ditemukan di Regional.")
reg_raw["Wilayah"] = reg_raw["Wilayah"].astype("string").str.strip()
if reg_raw["Wilayah"].isna().any() or reg_raw["Wilayah"].eq("").any():
    raise ValueError("Ada nama wilayah kosong di Regional.")
if reg_raw["Wilayah"].duplicated().any():
    raise ValueError("Ada nama wilayah duplikat di Regional.")
for col in PLATFORMS:
    reg_raw[col] = pd.to_numeric(reg_raw[col], errors="coerce")
if reg_raw[PLATFORMS].isna().any().any():
    raise ValueError("Ada nilai kosong atau nonnumerik di kolom platform Regional.")
reg = reg_raw.set_index("Wilayah")[PLATFORMS].copy()
if len(reg) != 34:
    raise ValueError(f"Jumlah wilayah {len(reg)}, sedangkan rancangan mengharapkan 34 provinsi.")
if not reg[PLATFORMS].apply(lambda col: col.between(0, 100).all()).all():
    raise ValueError("Ada nilai indeks Regional di luar rentang 0–100.")

# C. Seri analisis: TikTok Shop hanya sejak mulai beroperasi
series = {p: ts[p].copy() for p in PLATFORMS}
series["TikTok Shop"] = ts.loc[TIKTOK_START:, "TikTok Shop"].copy()

print("=" * 60)
print("LAPORAN VALIDASI DATASET")
print("Time Series:", ts.shape, "|", ts.index.min().date(), "hingga", ts.index.max().date())
print("Duplikat tanggal:", int(ts.index.duplicated().sum()))
print("Nilai kosong Time Series:", int(ts[PLATFORMS].isna().sum().sum()))
print("Regional:", reg.shape, "| duplikat wilayah:", int(reg.index.duplicated().sum()))
print("Nilai kosong Regional:", int(reg.isna().sum().sum()))
print("Periode analisis TikTok Shop:", series["TikTok Shop"].index.min().date(), "| n =", len(series["TikTok Shop"]))
print(); print("Contoh Time Series:")
display(ts[PLATFORMS].head())
print(); print("Contoh Regional:")
display(reg.head())
print(); print("Validasi persiapan selesai.")

# BAGIAN A — Analisis Tren Waktu

## A1. Statistik deskriptif dan visualisasi tren

In [ ]:
# Statistik dihitung dari periode analisis yang sesuai untuk tiap platform.
rows = []
for p, s in series.items():
    rows.append({
        "Platform": p,
        "n": s.count(),
        "mean": s.mean(),
        "std": s.std(),
        "min": s.min(),
        "max": s.max(),
        "CV (%)": (s.std() / s.mean() * 100) if s.mean() != 0 else np.nan,
        "Bulan puncak": s.idxmax().strftime("%b %Y")
    })
desc = pd.DataFrame(rows).set_index("Platform")
display(desc.round(2))

In [ ]:
fig, axes = plt.subplots(len(PLATFORMS), 1, figsize=(11, 11), sharex=True)
for ax, p in zip(axes, PLATFORMS):
    s = series[p]
    ax.plot(s.index, s, color=COLORS[p], alpha=.45, lw=1.2, label="Bulanan")
    ax.plot(s.index, s.rolling(12, center=True, min_periods=6).mean(), color=COLORS[p], lw=2.4, label="Rata-rata bergerak 12 bulan")
    for name, d in EVENTS.items():
        ax.axvline(pd.Timestamp(d), color="grey", ls="--", lw=1)
    ax.set_ylabel(p); ax.set_ylim(0, 105)
axes[0].legend(loc="lower left", ncol=2, fontsize=8)
axes[0].set_title("Search interest per platform (indeks 0–100, skala relatif masing-masing)")
plt.tight_layout(); savefig("A1_tren_per_platform.png"); plt.show()

In [ ]:
# Grafik gabungan untuk melihat pola; level antarplatform tidak boleh dibandingkan jika normalisasi terpisah.
fig, ax = plt.subplots(figsize=(11, 4.5))
for p in PLATFORMS:
    s = series[p]
    ax.plot(s.index, s.rolling(12, center=True, min_periods=6).mean(), label=p, color=COLORS[p], lw=2.2)
for name, d in EVENTS.items():
    ax.axvline(pd.Timestamp(d), color="grey", ls="--", lw=1)
    ax.text(pd.Timestamp(d), 101, name.split("(")[0].strip(), rotation=90, va="top", ha="right", fontsize=7, color="grey")
ax.set_title("Rata-rata bergerak 12 bulan search interest")
ax.set_ylabel("Indeks relatif"); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); savefig("A1_tren_gabungan.png"); plt.show()

## A2. Pertumbuhan tahunan (YoY) dan CAGR

In [ ]:
# Rata-rata tahunan; TikTok Shop pada 2021 hanya mencakup bulan aktif (Sep–Des).
annual = ts[PLATFORMS].groupby(ts.index.year).mean()
annual["TikTok Shop"] = series["TikTok Shop"].groupby(series["TikTok Shop"].index.year).mean()
display(annual.round(1))

yoy = annual.pct_change() * 100
display(yoy.round(1).rename_axis("Tahun").style.set_caption("Perubahan YoY rata-rata tahunan (%)"))

def cagr(first, last, n_years):
    if first <= 0 or last < 0 or n_years <= 0:
        return np.nan
    return ((last / first) ** (1 / n_years) - 1) * 100

cg, periods = {}, {}
for p in PLATFORMS:
    a = annual[p].dropna()
    # 2021 TikTok Shop bukan tahun penuh. CAGR dari rata-rata tahunan parsial akan menyesatkan.
    if p == "TikTok Shop":
        cg[p] = np.nan
        periods[p] = "Tidak dihitung: tahun awal parsial"
    else:
        cg[p] = cagr(a.iloc[0], a.iloc[-1], a.index[-1] - a.index[0])
        periods[p] = f"{a.index[0]}–{a.index[-1]}"
cagr_df = pd.DataFrame({"CAGR (%/tahun)": pd.Series(cg), "Periode": pd.Series(periods)})
display(cagr_df.round(2))

annual.plot(kind="bar", figsize=(10, 4), color=[COLORS[p] for p in PLATFORMS], width=.8)
plt.title("Rata-rata search interest tahunan")
plt.xticks(rotation=0); plt.ylabel("Indeks relatif")
plt.tight_layout(); savefig("A2_rata_tahunan.png"); plt.show()

print("Catatan: CAGR TikTok Shop tidak dihitung karena tahun 2021 hanya mencakup Sep–Des.")

## A3. Uji tren: Mann-Kendall dan Sen's slope
Uji nonparametrik yang sesuai untuk indeks terbatas 0–100 dan tidak mengasumsikan distribusi normal. `Sen's slope` = perubahan indeks per bulan.

In [ ]:
rows = []
for p, s in series.items():
    r = mk.original_test(s.values)
    rows.append({"Platform": p, "n": len(s), "Tren": r.trend, "p-value": r.p, "Kendall tau": r.Tau,
                 "Sen slope (indeks/bln)": r.slope, "Sen slope (indeks/thn)": r.slope * 12})
mk_df = pd.DataFrame(rows).set_index("Platform")
display(mk_df.round(4))

## A4. Pola musiman: dekomposisi STL
Memisahkan **tren**, **musiman** (periode 12 bulan), dan **residu**. Kekuatan musiman (*Fs*) mendekati 1 berarti musiman kuat (Wang et al.). Gunakan untuk melihat efek Ramadan/Lebaran, Harbolnas 11.11 dan 12.12.

In [ ]:
stl_res, strength = {}, {}
for p, s in series.items():
    res = STL(s, period=12, robust=True).fit()
    stl_res[p] = res
    strength[p] = max(0, 1 - res.resid.var() / (res.seasonal + res.resid).var())
print("Kekuatan musiman (Fs):")
display(pd.Series(strength, name="Fs").round(3).to_frame())

fig, axes = plt.subplots(len(PLATFORMS), 3, figsize=(14, 11), sharex="col")
for i, p in enumerate(PLATFORMS):
    r = stl_res[p]
    axes[i, 0].plot(r.observed.index, r.observed, color=COLORS[p]); axes[i, 0].plot(r.trend.index, r.trend, color="k", lw=1.5)
    axes[i, 1].plot(r.seasonal.index, r.seasonal, color=COLORS[p])
    axes[i, 2].plot(r.resid.index, r.resid, color="grey")
    axes[i, 0].set_ylabel(p)
for ax, t in zip(axes[0], ["Observasi & tren", "Komponen musiman", "Residu"]): ax.set_title(t)
plt.tight_layout(); savefig("A4_stl.png"); plt.show()

In [ ]:
# Indeks musiman rata-rata per bulan (dari komponen musiman STL)
bulan = ["Jan","Feb","Mar","Apr","Mei","Jun","Jul","Agu","Sep","Okt","Nov","Des"]
seas = pd.DataFrame({p: stl_res[p].seasonal.groupby(stl_res[p].seasonal.index.month).mean() for p in PLATFORMS})
seas.index = bulan

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(seas, cmap="RdBu_r", center=0, annot=True, fmt=".1f", cbar_kws={"label": "Deviasi musiman (poin indeks)"}, ax=ax)
ax.set_title("Indeks musiman rata-rata per bulan")
plt.tight_layout(); savefig("A4_indeks_musiman.png"); plt.show()

print("Bulan puncak musiman per platform:")
display(seas.idxmax().to_frame("Bulan tertinggi").join(seas.idxmin().to_frame("Bulan terendah")))

In [ ]:
# Heatmap bulan × tahun per platform. TikTok Shop sebelum Sep 2021 ditampilkan kosong.
fig, axes = plt.subplots(1, len(PLATFORMS), figsize=(18, 4.2), sharey=True)
for ax, p in zip(axes, PLATFORMS):
    s = ts[p].copy()
    if p == "TikTok Shop":
        s.loc[s.index < TIKTOK_START] = np.nan
    piv = s.to_frame("v").assign(Tahun=s.index.year, Bulan=s.index.month).pivot(index="Bulan", columns="Tahun", values="v")
    piv.index = bulan
    sns.heatmap(piv, cmap="YlOrRd", vmin=0, vmax=100, ax=ax, cbar=False, linewidths=.3)
    ax.set_title(p); ax.set_ylabel("")
plt.tight_layout(); savefig("A4_heatmap_bulan_tahun.png"); plt.show()

## A5. Perubahan struktural (*structural break*)
Dua pendekatan: (1) **deteksi otomatis** dengan PELT (ruptures) pada rata-rata bergerak 3 bulan untuk mengurangi *noise*, dan (2) **uji Chow** pada tanggal peristiwa yang ditentukan peneliti (`EVENTS`).

In [ ]:
def detect_breaks(s, pen_scale=2.0, min_size=6):
    x = s.rolling(3, center=True, min_periods=1).mean().values.reshape(-1, 1)
    pen = pen_scale * np.log(len(x)) * np.var(x)           # penalti ala BIC; naikkan pen_scale (mis. 3) agar lebih konservatif, turunkan (1) agar lebih sensitif
    algo = rpt.Pelt(model="l2", min_size=min_size, jump=1).fit(x)
    bks = algo.predict(pen=pen)[:-1]                        # buang titik akhir
    return [s.index[b] for b in bks]

breaks = {p: detect_breaks(s) for p, s in series.items()}
for p, b in breaks.items():
    print(f"{p:12s}:", [d.strftime("%b %Y") for d in b] if b else "tidak ada perubahan terdeteksi")

fig, axes = plt.subplots(len(PLATFORMS), 1, figsize=(11, 11), sharex=True)
for ax, p in zip(axes, PLATFORMS):
    s = series[p]
    ax.plot(s.index, s, color=COLORS[p], alpha=.7)
    for d in breaks[p]: ax.axvline(d, color="crimson", ls="--")
    # rata-rata per segmen
    edges = [s.index[0]] + breaks[p] + [s.index[-1] + pd.offsets.MonthBegin(1)]
    for a, b in zip(edges[:-1], edges[1:]):
        seg = s[(s.index >= a) & (s.index < b)]
        ax.hlines(seg.mean(), a, b, color="k", lw=2)
    ax.set_ylabel(p); ax.set_ylim(0, 105)
axes[0].set_title("Perubahan struktural otomatis (PELT) – garis hitam = rata-rata segmen")
plt.tight_layout(); savefig("A5_structural_break.png"); plt.show()

In [ ]:
def chow_test(y, bk_date):
    """Uji Chow sederhana untuk perubahan intersep dan slope tren linear pada tanggal yang ditentukan."""
    y = y.dropna()
    t = np.arange(len(y), dtype=float)
    bk_date = pd.Timestamp(bk_date)
    i = y.index.searchsorted(bk_date)
    k = 2
    if i < 6 or len(y) - i < 6:
        return np.nan, np.nan

    def ssr(tt, yy):
        X = np.column_stack([np.ones_like(tt), tt])
        beta = np.linalg.lstsq(X, yy, rcond=None)[0]
        return float(np.sum((yy - X @ beta) ** 2))

    ssr_pooled = ssr(t, y.to_numpy(dtype=float))
    ssr_split = ssr(t[:i], y.to_numpy(dtype=float)[:i]) + ssr(t[i:], y.to_numpy(dtype=float)[i:])
    df_den = len(y) - 2 * k
    if df_den <= 0 or ssr_split <= 0:
        return np.nan, np.nan
    F = max(0.0, (ssr_pooled - ssr_split) / k) / (ssr_split / df_den)
    p_value = stats.f.sf(F, k, df_den)
    return F, p_value

rows = []
for ev, d in EVENTS.items():
    for p, s in series.items():
        F, pv = chow_test(s, d)
        rows.append({"Peristiwa": ev, "Tanggal": d, "Platform": p, "F": F,
                     "p-value": pv, "Signifikan (α=5%)": (pv < .05) if pd.notna(pv) else False})
chow_df = pd.DataFrame(rows)
display(chow_df.round(4))
print("Interpretasi: hasil Chow menunjukkan perubahan bentuk tren pada tanggal uji; tidak membuktikan bahwa peristiwa menyebabkan perubahan.")

In [ ]:
# Perbandingan rata-rata sebelum vs sesudah peristiwa (Mann-Whitney U) – berguna untuk interpretasi substantif
rows = []
for ev, d in EVENTS.items():
    d = pd.Timestamp(d)
    for p, s in series.items():
        a, b = s[s.index < d], s[s.index >= d]
        if len(a) < 6 or len(b) < 6: continue
        u, pv = stats.mannwhitneyu(a, b, alternative="two-sided")
        rows.append({"Peristiwa": ev, "Platform": p, "Rata2 sebelum": a.mean(), "Rata2 sesudah": b.mean(),
                     "Selisih": b.mean() - a.mean(), "p-value": pv})
display(pd.DataFrame(rows).round(3))

## A6. Hubungan antarplatform
Korelasi dihitung pada **level** (kecenderungan bergerak bersama) dan pada **selisih bulanan** (*first difference*, menghilangkan efek tren). Periode dibatasi sejak `TIKTOK_START` agar kelima platform sebanding.

In [ ]:
common = ts.loc[TIKTOK_START:, PLATFORMS]
corr_lvl  = common.corr(method="spearman")
corr_diff = common.diff().dropna().corr(method="spearman")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, c, t in zip(axes, [corr_lvl, corr_diff], ["Spearman – level", "Spearman – selisih bulanan"]):
    sns.heatmap(c, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax, cbar=False)
    ax.set_title(t)
plt.tight_layout(); savefig("A6_korelasi.png"); plt.show()

In [ ]:
# Cross-correlation pada selisih bulanan: apakah naik/turunnya satu platform mendahului platform lain?
def ccf_table(x, y, max_lag=6):
    dx, dy = x.diff().dropna(), y.diff().dropna()
    out = {}
    for lag in range(-max_lag, max_lag + 1):
        out[lag] = dx.corr(dy.shift(-lag)) if lag != 0 else dx.corr(dy)   # lag>0: x mendahului y
    return pd.Series(out)

pairs = [("TikTok Shop", "Tokopedia"), ("TikTok Shop", "Shopee"), ("TikTok Shop", "Lazada"), ("Shopee", "Tokopedia")]
fig, axes = plt.subplots(1, len(pairs), figsize=(16, 3.6), sharey=True)
n = len(common) - 1; ci = 1.96 / np.sqrt(n)
for ax, (a, b) in zip(axes, pairs):
    cc = ccf_table(common[a], common[b])
    ax.bar(cc.index, cc.values, color=np.where(cc.abs() > ci, "crimson", "lightgrey"))
    ax.axhline(ci, ls="--", c="k", lw=.8); ax.axhline(-ci, ls="--", c="k", lw=.8)
    ax.set_title(f"{a} → {b}", fontsize=10); ax.set_xlabel("Lag (bulan); lag>0: kolom pertama mendahului")
axes[0].set_ylabel("Korelasi")
plt.tight_layout(); savefig("A6_crosscorr.png"); plt.show()

# BAGIAN B — Analisis Regional (34 Provinsi)

> Data regional adalah satu potret agregat (tidak berdimensi waktu) dan setiap platform dinormalisasi sendiri (nilai 100 = provinsi dengan minat tertinggi **untuk platform itu**). Maka bandingkan *posisi relatif provinsi dalam satu platform*, bukan level antarplatform.

## B1. Statistik deskriptif dan peringkat provinsi

In [ ]:
display(reg.describe().T.round(2))

top, bottom = {}, {}
for p in PLATFORMS:
    s = reg[p].sort_values(ascending=False)
    top[p]    = [f"{i} ({v:.0f})" for i, v in s.head(5).items()]
    bottom[p] = [f"{i} ({v:.0f})" for i, v in s.tail(5)[::-1].items()]
print("5 provinsi TERTINGGI per platform"); display(pd.DataFrame(top, index=range(1, 6)))
print("5 provinsi TERENDAH per platform");  display(pd.DataFrame(bottom, index=range(1, 6)))

## B2. Visualisasi sebaran wilayah

In [ ]:
order = reg.mean(axis=1).sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(8, 11))
sns.heatmap(reg.loc[order], annot=True, fmt=".0f", cmap="YlOrRd", vmin=0, vmax=100, linewidths=.3,
            cbar_kws={"label": "Indeks (relatif per platform)"}, ax=ax)
ax.set_ylabel(""); ax.set_title("Search interest per provinsi dan platform\n(diurutkan menurut rata-rata)")
plt.tight_layout(); savefig("B2_heatmap_provinsi.png"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(PLATFORMS), figsize=(20, 9), sharey=False)
for ax, p in zip(axes, PLATFORMS):
    s = reg[p].sort_values()
    ax.barh(s.index, s.values, color=COLORS[p]); ax.set_title(p); ax.tick_params(axis="y", labelsize=7)
plt.tight_layout(); savefig("B2_ranking_provinsi.png"); plt.show()

In [ ]:
# OPSIONAL – Peta choropleth. Butuh geopandas dan file GeoJSON batas provinsi Indonesia
# (nama provinsi di GeoJSON harus disamakan dengan kolom 'Wilayah').
GEOJSON_PATH = None   # contoh: "indonesia_provinces.geojson"
if GEOJSON_PATH:
    import geopandas as gpd
    gdf = gpd.read_file(GEOJSON_PATH)
    NAME_COL = "PROVINSI"          # sesuaikan nama kolom provinsi di GeoJSON
    fig, axes = plt.subplots(1, len(PLATFORMS), figsize=(24, 5))
    for ax, p in zip(axes, PLATFORMS):
        gdf.merge(reg[[p]], left_on=NAME_COL, right_index=True, how="left").plot(
            column=p, cmap="YlOrRd", legend=True, ax=ax, edgecolor="white", linewidth=.3, missing_kwds={"color": "lightgrey"})
        ax.set_title(p); ax.axis("off")
    plt.tight_layout(); savefig("B2_choropleth.png"); plt.show()
else:
    print("Choropleth dilewati (GEOJSON_PATH belum diisi). Heatmap dan ranking di atas sudah mewakili sebaran wilayah.")

## B3. Platform dominan per provinsi

In [ ]:
dom = reg.idxmax(axis=1).rename("Platform dominan")
print("Jumlah provinsi per platform dominan:")
display(dom.value_counts().to_frame("Jumlah provinsi"))
display(dom.to_frame().join(reg.max(axis=1).rename("Nilai tertinggi")).sort_values("Platform dominan"))
print("Catatan: karena skala tiap platform relatif, 'dominan' berarti platform dengan indeks tertinggi di provinsi tsb "
      "pada skala masing-masing; gunakan hati-hati jika data belum dari satu kueri perbandingan.")

## B4. Variasi antarwilayah (koefisien variasi)

In [ ]:
var_df = pd.DataFrame({"Rata-rata": reg.mean(), "Std": reg.std(), "CV (%)": reg.std() / reg.mean() * 100,
                       "Rentang": reg.max() - reg.min(), "IQR": reg.quantile(.75) - reg.quantile(.25)})
display(var_df.round(2).sort_values("CV (%)"))
print("CV lebih tinggi = minat lebih timpang antarprovinsi; CV rendah = lebih merata.")

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=reg, palette=COLORS, ax=ax, width=.55); sns.stripplot(data=reg, color="k", size=3, alpha=.5, ax=ax)
ax.set_title("Sebaran search interest antarprovinsi per platform"); ax.set_ylabel("Indeks")
plt.tight_layout(); savefig("B4_boxplot.png"); plt.show()

## B5. Uji perbedaan antarplatform: Friedman + post-hoc Wilcoxon (Holm)
Lima platform diukur pada 34 provinsi yang sama (*repeated measures*) → uji Friedman. Ukuran efek: **Kendall's W** (0–1).

In [ ]:
chi2, p = stats.friedmanchisquare(*[reg[c] for c in PLATFORMS])
n, k = reg.shape[0], len(PLATFORMS)
W = chi2 / (n * (k - 1))
print(f"Friedman χ²({k-1}) = {chi2:.2f}, p = {p:.4g}, Kendall's W = {W:.3f}")

from itertools import combinations
from statsmodels.stats.multitest import multipletests
rows = []
for a, b in combinations(PLATFORMS, 2):
    st, pv = stats.wilcoxon(reg[a], reg[b])
    rows.append({"Pasangan": f"{a} vs {b}", "Median A": reg[a].median(), "Median B": reg[b].median(), "W": st, "p": pv})
ph = pd.DataFrame(rows)
ph["p Holm"] = multipletests(ph["p"], method="holm")[1]
ph["Signifikan (α=5%)"] = ph["p Holm"] < .05
display(ph.round(4))

## B6. Jawa vs luar Jawa (Mann-Whitney U)

In [ ]:
JAWA = ["Banten", "DKI Jakarta", "Jawa Barat", "Jawa Tengah", "DI Yogyakarta", "Jawa Timur"]
missing = [j for j in JAWA if j not in reg.index]
if missing: print("PERINGATAN, nama tidak ditemukan:", missing)
is_jawa = reg.index.isin(JAWA)

rows = []
for pf in PLATFORMS:
    a, b = reg.loc[is_jawa, pf], reg.loc[~is_jawa, pf]
    u, pv = stats.mannwhitneyu(a, b, alternative="two-sided")
    rbc = 2 * u / (len(a) * len(b)) - 1          # rank-biserial correlation (ukuran efek)
    rows.append({"Platform": pf, "Median Jawa": a.median(), "Median luar Jawa": b.median(),
                 "U": u, "p-value": pv, "Rank-biserial r": rbc})
jv = pd.DataFrame(rows).set_index("Platform")
jv["p Holm"] = multipletests(jv["p-value"], method="holm")[1]
display(jv.round(4))

melt = reg.assign(Wilayah=np.where(is_jawa, "Jawa", "Luar Jawa")).melt(id_vars="Wilayah", var_name="Platform", value_name="Indeks")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=melt, x="Platform", y="Indeks", hue="Wilayah", palette=["#4C72B0", "#DD8452"], ax=ax)
ax.set_title("Jawa vs luar Jawa"); plt.tight_layout(); savefig("B6_jawa_vs_luar.png"); plt.show()

## B7. Kesamaan pola antarplatform lintas provinsi

In [ ]:
corr_reg = reg.corr(method="spearman")
fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(corr_reg, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax)
ax.set_title("Korelasi Spearman antarplatform (lintas 34 provinsi)")
plt.tight_layout(); savefig("B7_korelasi_regional.png"); plt.show()

# p-value korelasi
pv = pd.DataFrame(index=PLATFORMS, columns=PLATFORMS, dtype=float)
for a in PLATFORMS:
    for b in PLATFORMS:
        pv.loc[a, b] = 0 if a == b else stats.spearmanr(reg[a], reg[b])[1]
display(pv.round(4).style.set_caption("p-value korelasi Spearman"))

# BAGIAN C — Ringkasan Terpadu dan Ekspor

In [ ]:
summary = pd.DataFrame(index=PLATFORMS)
summary["Tren (Mann-Kendall)"]      = mk_df["Tren"]
summary["Sen slope (indeks/thn)"]   = mk_df["Sen slope (indeks/thn)"].round(2)
summary["CAGR (%)"]                 = cagr_df["CAGR (%/tahun)"]
summary["Kekuatan musiman (Fs)"]    = pd.Series(strength).round(2)
summary["Bulan puncak musiman"]     = seas.idxmax()
summary["Break terdeteksi"]         = [", ".join(d.strftime("%b %Y") for d in breaks[p]) or "-" for p in PLATFORMS]
summary["Rata2 regional"]           = reg.mean().round(1)
summary["CV regional (%)"]          = var_df["CV (%)"].round(1)
summary["Provinsi tertinggi"]       = [reg[p].idxmax() for p in PLATFORMS]
summary["Jumlah provinsi dominan"]  = dom.value_counts().reindex(PLATFORMS).fillna(0).astype(int)
display(summary)

In [ ]:
xlsx_path = os.path.join(OUT_DIR, "hasil_analisis.xlsx")
with pd.ExcelWriter(xlsx_path) as w:
    summary.to_excel(w, sheet_name="Ringkasan")
    desc.round(3).to_excel(w, sheet_name="A1_Deskriptif")
    annual.round(2).to_excel(w, sheet_name="A2_Tahunan")
    yoy.round(2).to_excel(w, sheet_name="A2_YoY")
    mk_df.round(4).to_excel(w, sheet_name="A3_MannKendall")
    seas.round(2).to_excel(w, sheet_name="A4_Musiman")
    chow_df.round(4).to_excel(w, sheet_name="A5_Chow", index=False)
    corr_lvl.round(3).to_excel(w, sheet_name="A6_Korelasi_Level")
    corr_diff.round(3).to_excel(w, sheet_name="A6_Korelasi_Diff")
    reg.describe().T.round(2).to_excel(w, sheet_name="B1_Deskriptif")
    var_df.round(2).to_excel(w, sheet_name="B4_Variasi")
    ph.round(4).to_excel(w, sheet_name="B5_PostHoc", index=False)
    jv.round(4).to_excel(w, sheet_name="B6_Jawa_vs_Luar")
    corr_reg.round(3).to_excel(w, sheet_name="B7_Korelasi")
print("Tersimpan di folder:", os.path.abspath(OUT_DIR))

## Panduan interpretasi untuk bab Hasil dan Pembahasan

- **Tren (A3):** laporkan arah, *p-value*, dan Sen's slope per tahun. Jangan membandingkan besar slope antarplatform bila skala belum sebanding.
- **Musiman (A4):** sebut bulan puncak dan kaitkan dengan kalender belanja (Ramadan/Lebaran, 11.11, 12.12, Harbolnas).
- **Break (A5):** hasil PELT bersifat eksploratif; uji Chow menguji tanggal yang Anda tentukan berdasarkan teori/peristiwa. Tampilkan keduanya.
- **Korelasi (A6):** korelasi bukan kausalitas. Korelasi negatif TikTok Shop–Tokopedia/Lazada hanya mengindikasikan pola bergerak berlawanan.
- **Regional (B):** gunakan untuk menggambarkan *di mana* minat relatif tinggi pada tiap platform; peringkat provinsi lebih aman ditafsirkan daripada level antarplatform.
- **Keterbatasan:** Google Trends adalah sampel pencarian, bukan penjualan; indeks relatif; hasil dapat sedikit berbeda tiap unduhan; kata kunci "Shopee" dsb. dapat mencerminkan pencarian navigasi, bukan niat membeli.